# Agent Trace Triage — Jev vs LLM Experiment

This notebook builds the **Agent Trace Triage** project: a small experiment that
uses TypeSafe's **Jev** (a "System One" decision model) as a fast triage layer over
AI agent execution traces, compares it head-to-head against a general LLM on the
same task, and then wires the two together into a **Jev → LLM cascade**.

**What you'll do:**
1. Install dependencies for OpenRouter (via `requests`) and (optionally) an LLM SDK for comparison
2. Load ~20 synthetic agent traces
3. Ask Jev four questions per trace: `anomalous` (Noul), `category` (Choice), `severity` (Score), `investigate` (Noul)
4. Run the same traces through an LLM with a matching JSON schema
5. Compare latency, cost, and agreement between the two
6. Build a Jev → LLM cascade (fast filter, then deep analysis only on flagged traces)
7. Print a tiny terminal-style dashboard

> **Note on access:** This notebook calls Jev through **OpenRouter**'s
> TypeSafe-compatible System One endpoint, so no TypeSafe waitlist key is needed —
> just an `OPENROUTER_API_KEY`. **Caveat:** OpenRouter's own listing for this model
> is inconsistent as of writing (some pages say live, one says "coming soon") — see
> the note in section 2 before relying on it. This notebook also **runs in a MOCK
> mode by default** (no key needed at all) so you can see the full pipeline work
> end-to-end first. As soon as you add a real key in the setup cell, `USE_MOCK_JEV`
> flips to `False` automatically — no other code needs to change, since the mock
> mirrors the real client's interface.


## 1. Setup & installation

In [ ]:
# We call Jev through OpenRouter's TypeSafe-compatible System One endpoint,
# so we don't need the typesafe-sdk package -- plain HTTP via `requests` is enough.
!pip install -q requests

# Groq SDK for the LLM comparison arm (fast inference, OpenAI-compatible chat API).
!pip install -q groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.0 MB/s eta 0:00:00


In [ ]:
import os
import time
import json
import random
from dataclasses import dataclass, field
from typing import Any

# --- Configuration -----------------------------------------------------------
# Loads OPENROUTER_API_KEY and GROQ_API_KEY from (in order of preference):
#   1. Colab Secrets (the 🔑 icon in the left sidebar) -- recommended
#   2. Environment variables already set in this runtime
#   3. A manual, hidden getpass() prompt (works outside Colab too)
# Keys are never printed or written to the notebook file.
#
# OPENROUTER_API_KEY comes from your OpenRouter account (openrouter.ai/settings/keys)
# -- NOT from typesafe.ai. Using OpenRouter means no TypeSafe waitlist key is
# required (assuming the model is actually live there -- see the caveat above).

def _load_secret(name: str) -> str:
    val = os.environ.get(name, "")
    if val:
        return val
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            return val
    except Exception:
        pass
    from getpass import getpass
    val = getpass(f"Enter {name} (input hidden, leave blank to skip / use mock mode): ")
    if val:
        os.environ[name] = val
    return val

OPENROUTER_API_KEY = _load_secret("OPENROUTER_API_KEY")
GROQ_API_KEY = _load_secret("GROQ_API_KEY")

USE_MOCK_JEV = len(OPENROUTER_API_KEY) == 0
USE_MOCK_LLM = len(GROQ_API_KEY) == 0

print(f"USE_MOCK_JEV = {USE_MOCK_JEV}  ({'no key found -- using mock' if USE_MOCK_JEV else 'real OpenRouter key loaded'})")
print(f"USE_MOCK_LLM = {USE_MOCK_LLM}  ({'no key found -- using mock' if USE_MOCK_LLM else 'real Groq API key loaded'})")


USE_MOCK_JEV = False  (real OpenRouter key loaded)
USE_MOCK_LLM = False  (real Groq API key loaded)


## 2. The Jev client (OpenRouter, with a mock fallback)

We're calling Jev through **OpenRouter**'s System One endpoint instead of Vercel AI
Gateway. Per OpenRouter's own SDK docs, this endpoint is TypeSafe-compatible and
keeps the native request shape (`state` + `questions` with `choice`/`score`/`noul`
types), and bare model IDs like `jev-1.13` get mapped onto OpenRouter's
`typesafe/` namespace automatically.

```
POST https://openrouter.ai/api/v1/systemone
Authorization: Bearer $OPENROUTER_API_KEY
{
  "model": "typesafe/jev-1.13",
  "state": {...},
  "questions": {
    "billing": {"type": "noul", "instructions": "..."},
    "tone":    {"type": "choice", "instructions": "...", "criteria": {...}},
    "urgency": {"type": "score", "instructions": "...", "criteria": [...]}
  }
}
```

**Important caveat -- checked at the time this notebook was written:** OpenRouter's
own listing for this model is inconsistent. `openrouter.ai/typesafe/jev-1.13`
currently shows **"Coming soon, not yet available"**, while a dated variant of the
same model page and OpenRouter's Typesafe provider page both show it live with
pricing. This matches the "Listed in beta -- check the model page before you build
on it" warning from the original route comparison. **Check
[openrouter.ai/typesafe/jev-1.13](https://openrouter.ai/typesafe/jev-1.13) yourself
right before running this for real** -- if it's still "coming soon", the real call
below will fail with a clear error (not silently), and you'll want to fall back to
the mock, Vercel AI Gateway, or the TypeSafe direct waitlist instead.

As with the earlier versions, the mock client mirrors the same interface so the
rest of the notebook works identically in mock mode.


In [ ]:
@dataclass
class NoulResult:
    noul: float

@dataclass
class ChoiceResult:
    choice: str
    probabilities: dict = field(default_factory=dict)
    confidence: float = 0.0

@dataclass
class ScoreResult:
    score: float
    probabilities: dict = field(default_factory=dict)
    confidence: float = 0.0

@dataclass
class SystemOneResponse:
    nouls: dict
    choices: dict
    scores: dict


# --- Question type definitions (used to build the request body) --------------

@dataclass
class Noul:
    instructions: str

@dataclass
class Choice:
    instructions: str
    criteria: dict  # option_name -> description or None

@dataclass
class Score:
    instructions: str
    criteria: list  # ordered level labels


def _question_to_payload(q) -> dict:
    if isinstance(q, Noul):
        return {"type": "noul", "instructions": q.instructions}
    elif isinstance(q, Choice):
        return {"type": "choice", "instructions": q.instructions, "criteria": q.criteria}
    elif isinstance(q, Score):
        return {"type": "score", "instructions": q.instructions, "criteria": q.criteria}
    raise TypeError(f"Unknown question type: {type(q)}")


_printed_raw_response_once = False

def _parse_gateway_response(raw: dict) -> SystemOneResponse:
    """Best-effort parser for the Vercel AI Gateway TypeSafe passthrough response.

    Confirmed from Vercel's docs: the request keeps TypeSafe's native shape.
    NOT independently confirmed: the exact response field names for this specific
    endpoint. We try TypeSafe's documented direct-API convention first
    (top-level "nouls"/"choices"/"scores" dicts keyed by question name), then fall
    back to a flatter "answers" dict if that's what comes back instead.
    """
    global _printed_raw_response_once
    if not _printed_raw_response_once:
        print("--- Raw Vercel AI Gateway response (first call only, for verification) ---")
        print(json.dumps(raw, indent=2)[:2000])
        print("--- end raw response ---\n")
        _printed_raw_response_once = True

    nouls, choices, scores = {}, {}, {}

    if "nouls" in raw or "choices" in raw or "scores" in raw:
        for k, v in raw.get("nouls", {}).items():
            nouls[k] = NoulResult(noul=v["noul"] if isinstance(v, dict) else v)
        for k, v in raw.get("choices", {}).items():
            choices[k] = ChoiceResult(choice=v["choice"], probabilities=v.get("probabilities", {}), confidence=v.get("confidence", 0.0))
        for k, v in raw.get("scores", {}).items():
            scores[k] = ScoreResult(score=v["score"], probabilities=v.get("probabilities", {}), confidence=v.get("confidence", 0.0))
    elif "answers" in raw:
        for k, v in raw["answers"].items():
            t = v.get("type")
            if t == "noul":
                nouls[k] = NoulResult(noul=v.get("noul", v.get("value")))
            elif t == "choice":
                choices[k] = ChoiceResult(choice=v.get("choice", v.get("value")), probabilities=v.get("probabilities", {}), confidence=v.get("confidence", 0.0))
            elif t == "score":
                scores[k] = ScoreResult(score=v.get("score", v.get("value")), probabilities=v.get("probabilities", {}), confidence=v.get("confidence", 0.0))
    else:
        raise ValueError(
            "Unrecognized response shape from Vercel AI Gateway -- inspect the raw "
            "response printed above and update _parse_gateway_response to match it."
        )

    return SystemOneResponse(nouls=nouls, choices=choices, scores=scores)


class OpenRouterJevClient:
    """Calls Jev through OpenRouter's TypeSafe-compatible System One endpoint.

    NOTE: as of writing, OpenRouter's own model page for typesafe/jev-1.13 is
    inconsistent about availability (see markdown cell above). If the model
    truly isn't live yet, this will raise a RuntimeError with OpenRouter's
    error body -- read that message before assuming something else is wrong.
    """

    ENDPOINT = "https://openrouter.ai/api/v1/systemone"

    def __init__(self, api_key: str, model: str = "typesafe/jev-1.13"):
        self.api_key = api_key
        self.model = model

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        return False

    def system_one(self, state, questions: dict) -> SystemOneResponse:
        import requests
        body = {
            "model": self.model,
            "state": state,
            "questions": {k: _question_to_payload(q) for k, q in questions.items()},
        }
        resp = requests.post(
            self.ENDPOINT,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
                # Optional but recommended by OpenRouter for attribution/rankings;
                # harmless to leave as-is or edit for your own app.
                "HTTP-Referer": "https://github.com/",
                "X-Title": "Agent Trace Triage notebook",
            },
            json=body,
            timeout=30,
        )
        if resp.status_code >= 400:
            raise RuntimeError(
                f"OpenRouter System One call failed ({resp.status_code}). "
                f"If this says the model is unavailable, check "
                f"https://openrouter.ai/typesafe/jev-1.13 for current status "
                f"before retrying. Response body: {resp.text[:500]}"
            )
        return _parse_gateway_response(resp.json())


class MockJevClient:
    """Deterministic-ish mock of the Jev client for offline development.

    Heuristics are simple and transparent on purpose -- this is a stand-in so you
    can build and test the pipeline shape, NOT a substitute for the real model's
    judgment. Used automatically when OPENROUTER_API_KEY isn\'t set.
    """

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        return False

    def system_one(self, state: dict, questions: dict) -> SystemOneResponse:
        time.sleep(random.uniform(0.05, 0.25))

        nouls, choices, scores = {}, {}, {}

        errors = state.get("errors", 0)
        retries = state.get("retries", 0)
        duration_ms = state.get("duration_ms", 0)
        tool_errors = state.get("tool_errors", [])

        anomaly_signal = errors > 0 or retries > 1 or duration_ms > 8000 or bool(tool_errors)

        for key, q in questions.items():
            if isinstance(q, Noul):
                if "anomal" in key or "anomal" in q.instructions.lower():
                    val = 0.85 if anomaly_signal else 0.08
                elif "investigate" in key or "investigate" in q.instructions.lower():
                    val = 0.9 if (anomaly_signal and (errors > 1 or duration_ms > 8000)) else (0.4 if anomaly_signal else 0.05)
                else:
                    val = 0.5
                nouls[key] = NoulResult(noul=round(val + random.uniform(-0.03, 0.03), 3))

            elif isinstance(q, Choice):
                options = list(q.criteria.keys())
                if errors > 0 and tool_errors:
                    pick = "tool_failure"
                elif duration_ms > 8000:
                    pick = "latency"
                elif retries > 1:
                    pick = "retries"
                elif state.get("tokens", 0) > 20000:
                    pick = "token_usage"
                else:
                    pick = "normal"
                pick = pick if pick in options else random.choice(options)
                probs = {opt: (0.7 if opt == pick else 0.3 / max(1, len(options) - 1)) for opt in options}
                choices[key] = ChoiceResult(choice=pick, probabilities=probs, confidence=round(probs[pick], 3))

            elif isinstance(q, Score):
                levels = q.criteria
                n = len(levels)
                if anomaly_signal:
                    idx = min(n - 1, 2 + (1 if errors > 1 else 0) + (1 if duration_ms > 10000 else 0))
                else:
                    idx = 0
                val = idx + random.uniform(-0.2, 0.2)
                val = max(0, min(n - 1, val))
                scores[key] = ScoreResult(score=round(val, 2), confidence=round(0.6 + random.uniform(0, 0.3), 3))

        return SystemOneResponse(nouls=nouls, choices=choices, scores=scores)


def get_jev_client():
    if USE_MOCK_JEV:
        return MockJevClient()
    else:
        return OpenRouterJevClient(api_key=os.environ["OPENROUTER_API_KEY"])


## 3. Synthetic agent traces

Twenty fake traces standing in for real execution logs from an agent (support bot, research agent, etc.).

In [ ]:
TRACES = [
    {"trace_id": "trace-001", "agent": "support",  "duration_ms": 850,   "tool_calls": 3,  "errors": 0, "retries": 0, "tokens": 3200,  "final_answer": "Your refund has been processed.", "tool_errors": []},
    {"trace_id": "trace-002", "agent": "support",  "duration_ms": 12800, "tool_calls": 12, "errors": 3, "retries": 5, "tokens": 28000, "final_answer": "I was unable to complete your request.", "tool_errors": ["stripe_timeout", "stripe_timeout", "db_lock"]},
    {"trace_id": "trace-003", "agent": "research",  "duration_ms": 4500,  "tool_calls": 8,  "errors": 0, "retries": 0, "tokens": 18000, "final_answer": "Here is the research summary...", "tool_errors": []},
    {"trace_id": "trace-004", "agent": "support",  "duration_ms": 9100,  "tool_calls": 6,  "errors": 1, "retries": 2, "tokens": 9800,  "final_answer": "Let me check on that for you.", "tool_errors": ["stripe_timeout"]},
    {"trace_id": "trace-005", "agent": "support",  "duration_ms": 620,   "tool_calls": 2,  "errors": 0, "retries": 0, "tokens": 1900,  "final_answer": "Your order ships tomorrow.", "tool_errors": []},
    {"trace_id": "trace-006", "agent": "research",  "duration_ms": 21000, "tool_calls": 15, "errors": 0, "retries": 0, "tokens": 42000, "final_answer": "Full literature review attached.", "tool_errors": []},
    {"trace_id": "trace-007", "agent": "support",  "duration_ms": 1100,  "tool_calls": 4,  "errors": 0, "retries": 0, "tokens": 4200,  "final_answer": "I've updated your address.", "tool_errors": []},
    {"trace_id": "trace-008", "agent": "billing",   "duration_ms": 15300, "tool_calls": 9,  "errors": 4, "retries": 6, "tokens": 15400, "final_answer": "Error processing payment.", "tool_errors": ["stripe_timeout", "stripe_timeout", "auth_failed"]},
    {"trace_id": "trace-009", "agent": "support",  "duration_ms": 780,   "tool_calls": 3,  "errors": 0, "retries": 0, "tokens": 2800,  "final_answer": "Password reset link sent.", "tool_errors": []},
    {"trace_id": "trace-010", "agent": "research",  "duration_ms": 5200,  "tool_calls": 7,  "errors": 0, "retries": 1, "tokens": 16500, "final_answer": "Summary compiled from 12 sources.", "tool_errors": []},
    {"trace_id": "trace-011", "agent": "billing",   "duration_ms": 2400,  "tool_calls": 5,  "errors": 0, "retries": 0, "tokens": 6100,  "final_answer": "Invoice #4521 sent.", "tool_errors": []},
    {"trace_id": "trace-012", "agent": "support",  "duration_ms": 18700, "tool_calls": 20,  "errors": 2, "retries": 4, "tokens": 51000, "final_answer": "I'm still working on this, please hold.", "tool_errors": ["db_lock"]},
    {"trace_id": "trace-013", "agent": "billing",   "duration_ms": 950,   "tool_calls": 3,  "errors": 0, "retries": 0, "tokens": 3400,  "final_answer": "Refund of $42.00 issued.", "tool_errors": []},
    {"trace_id": "trace-014", "agent": "research",  "duration_ms": 3300,  "tool_calls": 6,  "errors": 0, "retries": 0, "tokens": 12000, "final_answer": "Three relevant papers found.", "tool_errors": []},
    {"trace_id": "trace-015", "agent": "support",  "duration_ms": 720,   "tool_calls": 2,  "errors": 0, "retries": 0, "tokens": 2100,  "final_answer": "Ticket closed as resolved.", "tool_errors": []},
    {"trace_id": "trace-016", "agent": "billing",   "duration_ms": 11200, "tool_calls": 10, "errors": 2, "retries": 3, "tokens": 19800, "final_answer": "Charge disputed, escalating.", "tool_errors": ["auth_failed"]},
    {"trace_id": "trace-017", "agent": "support",  "duration_ms": 1450,  "tool_calls": 4,  "errors": 0, "retries": 0, "tokens": 5300,  "final_answer": "Subscription cancelled.", "tool_errors": []},
    {"trace_id": "trace-018", "agent": "research",  "duration_ms": 26500, "tool_calls": 18, "errors": 1, "retries": 2, "tokens": 61000, "final_answer": "Partial results, some sources unreachable.", "tool_errors": ["fetch_timeout"]},
    {"trace_id": "trace-019", "agent": "support",  "duration_ms": 900,   "tool_calls": 3,  "errors": 0, "retries": 0, "tokens": 3000,  "final_answer": "Your account has been verified.", "tool_errors": []},
    {"trace_id": "trace-020", "agent": "billing",   "duration_ms": 8600,  "tool_calls": 7,  "errors": 1, "retries": 2, "tokens": 11000, "final_answer": "Payment method updated after retry.", "tool_errors": ["stripe_timeout"]},
]

with open("traces.json", "w") as f:
    json.dump(TRACES, f, indent=2)

print(f"Loaded {len(TRACES)} traces")
TRACES[1]


Loaded 20 traces


{'trace_id': 'trace-002',
 'agent': 'support',
 'duration_ms': 12800,
 'tool_calls': 12,
 'errors': 3,
 'retries': 5,
 'tokens': 28000,
 'final_answer': 'I was unable to complete your request.',
 'tool_errors': ['stripe_timeout', 'stripe_timeout', 'db_lock']}

## 4. Experiment A — Jev as the triage layer

Ask all four questions in a single `system_one()` call per trace, and record latency + decisions.

In [ ]:
JEV_QUESTIONS = {
    "anomalous": Noul(
        instructions="Does this agent trace show meaningful anomalous behavior?"
    ),
    "category": Choice(
        instructions="What is the primary problem category in this trace?",
        criteria={
            "latency": None,
            "tool_failure": None,
            "retries": None,
            "token_usage": None,
            "quality": None,
            "normal": None,
        },
    ),
    "severity": Score(
        instructions="How severe is this trace?",
        criteria=["normal", "minor", "moderate", "high", "critical"],
    ),
    "investigate": Noul(
        instructions="Should an engineer investigate this trace?"
    ),
}


def analyze_trace_with_jev(trace: dict) -> dict:
    start = time.perf_counter()
    with get_jev_client() as client:
        response = client.system_one(state=trace, questions=JEV_QUESTIONS)
    elapsed_ms = (time.perf_counter() - start) * 1000

    return {
        "trace_id": trace["trace_id"],
        "anomalous": response.nouls["anomalous"].noul,
        "category": response.choices["category"].choice,
        "category_confidence": response.choices["category"].confidence,
        "severity": response.scores["severity"].score,
        "investigate": response.nouls["investigate"].noul,
        "latency_ms": round(elapsed_ms, 1),
        # Jev's published pricing: ~$0.042 per million input tokens, output free.
        # We approximate input tokens as a rough char/4 estimate of state+questions.
        "est_cost_usd": round((len(json.dumps(trace)) / 4) * (0.042 / 1_000_000), 8),
    }


jev_results = [analyze_trace_with_jev(t) for t in TRACES]

import pandas as pd
jev_df = pd.DataFrame(jev_results)
jev_df


--- Raw Vercel AI Gateway response (first call only, for verification) ---
{
  "model": "typesafe/jev-1.13-20260917",
  "answers": {
    "anomalous": {
      "type": "noul",
      "noul": 0.08
    },
    "category": {
      "type": "choice",
      "choice": "normal",
      "probabilities": {
        "normal": 0.99,
        "latency": 0,
        "token_usage": 0.01,
        "retries": 0,
        "quality": 0,
        "tool_failure": 0
      },
      "confidence": 0.99
    },
    "severity": {
      "type": "score",
      "score": 0,
      "legend": {
        "0": "normal",
        "1": "minor",
        "2": "moderate",
        "3": "high",
        "4": "critical"
      },
      "probabilities": {
        "0": 1,
        "1": 0,
        "2": 0,
        "3": 0,
        "4": 0
      },
      "confidence": 1
    },
    "investigate": {
      "type": "noul",
      "noul": 0.14
    }
  },
  "usage": {
    "input_tokens": 482,
    "output_tokens": 114,
    "cost": 2.0244e-05
  },
  "id": "gen-

,trace_id,anomalous,category,category_confidence,severity,investigate,latency_ms,est_cost_usd
0,trace-001,0.08,normal,0.99,0.00,0.14,349.3,0.000002
1,trace-002,0.80,tool_failure,0.99,3.12,0.91,201.6,0.000003
2,trace-003,0.10,normal,0.94,0.01,0.24,194.2,0.000002
3,trace-004,0.48,tool_failure,0.90,1.79,0.75,202.6,0.000002
4,trace-005,0.09,normal,0.98,0.00,0.15,243.7,0.000002
5,trace-006,0.14,normal,0.80,0.09,0.28,187.6,0.000002
6,trace-007,0.10,normal,0.98,0.01,0.17,192.4,0.000002
7,trace-008,0.83,tool_failure,0.99,3.12,0.91,199.4,0.000003
8,trace-009,0.08,normal,0.99,0.00,0.17,229.2,0.000002
9,trace-010,0.12,normal,0.58,0.39,0.36,203.0,0.000002


## 5. Experiment B — the same task, via an LLM

We ask a general-purpose LLM to return the identical JSON shape, so we can compare
apples to apples on latency, cost, and agreement. This uses **Groq** (fast LPU-based
inference, OpenAI-compatible chat API, running Qwen3.8 27B by default) as the
comparison model — swap `model=` in `real_llm_call` for any other Groq-hosted model.
(Note: `llama-3.3-70b-versatile` was decommissioned by Groq on 2026-08-16, which is
why this notebook no longer defaults to it.)

If no `GROQ_API_KEY` is set, this falls back to a mock LLM responder so the rest of
the notebook (the comparison table) still runs end-to-end.


In [ ]:
LLM_PROMPT_TEMPLATE = '''You are analyzing a single AI agent execution trace. Return ONLY valid JSON
(no markdown, no explanation) matching exactly this schema:

{{
  "anomalous": <boolean>,
  "category": <one of: "latency", "tool_failure", "retries", "token_usage", "quality", "normal">,
  "severity": <integer 0-4, where 0=normal, 1=minor, 2=moderate, 3=high, 4=critical>,
  "investigate": <boolean>
}}

Trace:
{trace_json}
'''

def mock_llm_call(trace: dict) -> dict:
    # Reuse the mock Jev heuristic under the hood so the comparison table has
    # something sensible to show even with no API keys configured at all.
    with MockJevClient() as client:
        resp = client.system_one(state=trace, questions=JEV_QUESTIONS)
    time.sleep(random.uniform(1.5, 4.0))  # simulate LLM's slower round trip
    return {
        "anomalous": resp.nouls["anomalous"].noul > 0.5,
        "category": resp.choices["category"].choice,
        "severity": round(resp.scores["severity"].score),
        "investigate": resp.nouls["investigate"].noul > 0.5,
    }


def real_llm_call(trace: dict, model: str = "qwen/qwen3.8-27b") -> dict:
    from groq import Groq
    client = Groq()  # reads GROQ_API_KEY from the environment
    prompt = LLM_PROMPT_TEMPLATE.format(trace_json=json.dumps(trace, indent=2))
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.6,
        max_completion_tokens=2048,
        top_p=0.95,
        reasoning_effort="default",
        stream=False,  # non-streaming so we can parse the full JSON response at once
        stop=None,
    )
    text = completion.choices[0].message.content.strip()
    # Strip markdown fences if the model adds them despite instructions
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)


_llm_error_printed_count = 0

def analyze_trace_with_llm(trace: dict) -> dict:
    global _llm_error_printed_count
    start = time.perf_counter()
    error_text = None
    try:
        if USE_MOCK_LLM:
            parsed = mock_llm_call(trace)
            json_ok = True
        else:
            parsed = real_llm_call(trace)
            json_ok = True
    except Exception as e:
        parsed = {"anomalous": None, "category": None, "severity": None, "investigate": None}
        json_ok = False
        error_text = f"{type(e).__name__}: {e}"
        # Print the first few real errors in full so you can actually diagnose
        # what's failing, instead of silently getting all-None rows.
        if _llm_error_printed_count < 3:
            print(f"[analyze_trace_with_llm] {trace['trace_id']} failed -- {error_text}")
            _llm_error_printed_count += 1
    elapsed_ms = (time.perf_counter() - start) * 1000

    return {
        "trace_id": trace["trace_id"],
        "anomalous": parsed.get("anomalous"),
        "category": parsed.get("category"),
        "severity": parsed.get("severity"),
        "investigate": parsed.get("investigate"),
        "latency_ms": round(elapsed_ms, 1),
        "json_ok": json_ok,
        "error": error_text,
        # Rough placeholder cost -- update these per-token rates to match Groq's current
        # published pricing for qwen/qwen3.8-27b (check console.groq.com/docs/models).
        "est_cost_usd": round((len(json.dumps(trace)) / 4) * (0.59 / 1_000_000) + (50 / 4) * (0.79 / 1_000_000), 8),
    }


llm_results = [analyze_trace_with_llm(t) for t in TRACES]
llm_df = pd.DataFrame(llm_results)
llm_df


[analyze_trace_with_llm] trace-007 failed -- RateLimitError: Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.8-27b` in organization `org_01j6hss1h3en78zgdyn5xed9mc` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 1009. The request's expected output tokens exceed the enforced limit; reduce max_tokens (or the request's expected output) and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing", 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[analyze_trace_with_llm] trace-019 failed -- RateLimitError: Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.8-27b` in organization `org_01j6hss1h3en78zgdyn5xed9mc` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 1009. The request's expected output tokens exceed the enforced limit; reduce max_tokens (or the request's expected output) and try again. Need more tokens? Upg

,trace_id,anomalous,category,severity,investigate,latency_ms,json_ok,error,est_cost_usd
0,trace-001,False,normal,0.0,False,1952.3,True,None,0.000039
1,trace-002,True,tool_failure,3.0,True,396.4,True,None,0.000047
2,trace-003,False,normal,0.0,False,1165.0,True,None,0.000039
3,trace-004,True,quality,4.0,True,426.4,True,None,0.000041
4,trace-005,False,normal,0.0,False,303.0,True,None,0.000038
5,trace-006,False,normal,0.0,False,285.6,True,None,0.000040
6,trace-007,None,None,NaN,None,133.5,False,RateLimitError: Error code: 429 - {'error': {'...,0.000038
7,trace-008,True,tool_failure,3.0,True,709.3,True,None,0.000045
8,trace-009,False,normal,0.0,False,1452.3,True,None,0.000038
9,trace-010,False,normal,0.0,False,721.2,True,None,0.000039


## 6. Head-to-head comparison

In [ ]:
summary_rows = []

def pct(x):
    return f"{x*100:.1f}%"

jev_investigate = jev_df["investigate"] > 0.5
llm_investigate = llm_df["investigate"].fillna(False).astype(bool)
agreement = (jev_investigate.values == llm_investigate.values).mean()

summary = pd.DataFrame({
    "metric": [
        "Average latency (ms)",
        "p95 latency (ms)",
        "Total cost (USD, 20 traces)",
        "Avg cost per trace (USD)",
        "JSON/parse failures",
        "Flagged for investigation (count)",
    ],
    "Jev": [
        round(jev_df["latency_ms"].mean(), 1),
        round(jev_df["latency_ms"].quantile(0.95), 1),
        round(jev_df["est_cost_usd"].sum(), 6),
        round(jev_df["est_cost_usd"].mean(), 8),
        0,  # Jev's typed outputs can't fail to parse by construction
        int(jev_investigate.sum()),
    ],
    "LLM": [
        round(llm_df["latency_ms"].mean(), 1),
        round(llm_df["latency_ms"].quantile(0.95), 1),
        round(llm_df["est_cost_usd"].sum(), 6),
        round(llm_df["est_cost_usd"].mean(), 8),
        int((~llm_df["json_ok"]).sum()),
        int(llm_investigate.sum()),
    ],
})

print(f"Agreement between Jev and LLM on 'should investigate': {pct(agreement)}\n")
summary


Agreement between Jev and LLM on 'should investigate': 100.0%



/tmp/ipykernel_2805/1224975542.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  llm_investigate = llm_df["investigate"].fillna(False).astype(bool)


,metric,Jev,LLM
0,Average latency (ms),221.000000,583.800000
1,p95 latency (ms),319.900000,1477.300000
2,"Total cost (USD, 20 traces)",0.000043,0.000798
3,Avg cost per trace (USD),0.000002,0.000040
4,JSON/parse failures,0.000000,2.000000
5,Flagged for investigation (count),7.000000,7.000000


In [ ]:
# Side-by-side per-trace view
compare_df = jev_df[["trace_id", "category", "severity", "investigate", "latency_ms"]].merge(
    llm_df[["trace_id", "category", "severity", "investigate", "latency_ms"]],
    on="trace_id",
    suffixes=("_jev", "_llm"),
)
compare_df


,trace_id,category_jev,severity_jev,investigate_jev,latency_ms_jev,category_llm,severity_llm,investigate_llm,latency_ms_llm
0,trace-001,normal,0.00,0.14,349.3,normal,0.0,False,1952.3
1,trace-002,tool_failure,3.12,0.91,201.6,tool_failure,3.0,True,396.4
2,trace-003,normal,0.01,0.24,194.2,normal,0.0,False,1165.0
3,trace-004,tool_failure,1.79,0.75,202.6,quality,4.0,True,426.4
4,trace-005,normal,0.00,0.15,243.7,normal,0.0,False,303.0
5,trace-006,normal,0.09,0.28,187.6,normal,0.0,False,285.6
6,trace-007,normal,0.01,0.17,192.4,None,NaN,None,133.5
7,trace-008,tool_failure,3.12,0.91,199.4,tool_failure,3.0,True,709.3
8,trace-009,normal,0.00,0.17,229.2,normal,0.0,False,1452.3
9,trace-010,normal,0.39,0.36,203.0,normal,0.0,False,721.2


## 7. The Jev → LLM cascade

The interesting production pattern: use Jev as a cheap, fast filter over *every*
trace, and only spend LLM tokens on the small subset it flags for investigation.


In [ ]:
def cascade(traces):
    results = []
    llm_calls = 0
    for trace in traces:
        jev_result = analyze_trace_with_jev(trace)
        entry = {"trace_id": trace["trace_id"], **jev_result}

        if jev_result["investigate"] > 0.5:
            llm_calls += 1
            deep = analyze_trace_with_llm(trace)  # in a real cascade: send trace + ask for a root-cause explanation
            entry["llm_deep_dive"] = deep
        else:
            entry["llm_deep_dive"] = None

        results.append(entry)
    return results, llm_calls


cascade_results, llm_call_count = cascade(TRACES)

print(f"Traces processed by Jev: {len(TRACES)}")
print(f"Traces escalated to the LLM: {llm_call_count} ({llm_call_count/len(TRACES)*100:.0f}%)")
print(f"LLM calls avoided vs. running the LLM on everything: {len(TRACES) - llm_call_count}")


Traces processed by Jev: 20
Traces escalated to the LLM: 7 (35%)
LLM calls avoided vs. running the LLM on everything: 13


## 8. Tiny terminal-style dashboard

In [ ]:
def render_dashboard(results):
    total = len(results)
    anomalous = sum(1 for r in results if r["anomalous"] > 0.5)
    investigate = sum(1 for r in results if r["investigate"] > 0.5)

    print("╭" + "─" * 50 + "╮")
    print("│" + " Agent Trace Triage ".center(50) + "│")
    print("├" + "─" * 50 + "┤")
    print(f"│ Total traces:       {total:<33}│")
    print(f"│ Anomalous:          {anomalous:<33}│")
    print(f"│ Need investigation: {investigate:<33}│")
    print("├──────────┬─────────────┬──────────┬──────────────┤")
    print("│ Trace    │ Category    │ Severity │ Investigate  │")
    print("├──────────┼─────────────┼──────────┼──────────────┤")
    for r in results:
        tid = r["trace_id"][-7:]
        cat = r["category"][:11]
        sev = f"{r['severity']:.1f}"
        inv = "YES" if r["investigate"] > 0.5 else "no"
        print(f"│ {tid:<8} │ {cat:<11} │ {sev:<8} │ {inv:<12} │")
    print("╰──────────┴─────────────┴──────────┴──────────────╯")

render_dashboard(jev_results)


╭──────────────────────────────────────────────────╮
│                Agent Trace Triage                │
├──────────────────────────────────────────────────┤
│ Total traces:       20                               │
│ Anomalous:          4                                │
│ Need investigation: 7                                │
├──────────┬─────────────┬──────────┬──────────────┤
│ Trace    │ Category    │ Severity │ Investigate  │
├──────────┼─────────────┼──────────┼──────────────┤
│ ace-001  │ normal      │ 0.0      │ no           │
│ ace-002  │ tool_failur │ 3.1      │ YES          │
│ ace-003  │ normal      │ 0.0      │ no           │
│ ace-004  │ tool_failur │ 1.8      │ YES          │
│ ace-005  │ normal      │ 0.0      │ no           │
│ ace-006  │ normal      │ 0.1      │ no           │
│ ace-007  │ normal      │ 0.0      │ no           │
│ ace-008  │ tool_failur │ 3.1      │ YES          │
│ ace-009  │ normal      │ 0.0      │ no           │
│ ace-010  │ normal      │ 0.4    